# 02 Data Cleaning

Goal: turn the raw transactions into a clean, analysis-ready dataset 
based on the rules established during EDA (notebook 01).

The actual cleaning logic lives in `src/preprocessing.py`. This 
notebook **orchestrates** that logic, verifies the row-count audit 
trail, and saves the cleaned outputs as parquet for fast loading in 
downstream notebooks.

Keeping logic in `src/` and orchestration in notebooks means: one 
source of truth, easily testable, importable by the Streamlit app — 
not a one-off script that drifts out of sync with the analysis.

## 1 Setup

Import the cleaning functions from `src.preprocessing`. The 
`sys.path.insert(0, '..')` lets us import from the project root 
since this notebook lives in `notebooks/`.

In [7]:
import sys
sys.path.insert(0, '..')  # let us import from src/

from src.preprocessing import load_raw_data, clean_transactions

## 2 Load the raw data

Load via `load_raw_data()` from `src.preprocessing`. This concatenates 
both Excel sheets (2009-2010 and 2010-2011) into one DataFrame. Takes 
30-60 seconds — this is the slowest step in the entire pipeline because 
Excel parsing is expensive. Everything downstream uses parquet.

In [8]:
df = load_raw_data("../data/raw/online_retail_II.xlsx")
print(f"Raw shape: {df.shape}")

Raw shape: (1067371, 8)


## 3 Apply the cleaning pipeline

`clean_transactions()` applies the rules established in EDA, in order:

1. Drop rows with missing Customer ID
2. Drop cancellation invoices (Invoice starts with "C")
3. Drop non-positive Quantity (defensive — should be redundant after #1+#2)
4. Drop non-positive Price (administrative rows)
5. Drop non-product stock codes (POST, BANK CHARGES, AMAZONFEE, etc.)
6. Filter to UK
7. Cast Customer ID to int (no more NaN, safe to cast)
8. Add Revenue column (Quantity × Price)
9. Normalize string columns (Invoice and StockCode → str, for parquet compatibility)
10. Split wholesale (>60 orders OR >£20k spend) from retail
11. Drop retail customers with non-positive total spend

The verbose log below shows row counts after each step — this is the 
audit trail for any "where did rows X go?" questions later.

In [9]:
retail, wholesale = clean_transactions(df)

Starting cleaning: 1,067,371 rows
  Drop missing Customer ID: 1,067,371 -> 824,364 (dropped 243,007)
  Drop cancellation invoices: 824,364 -> 805,620 (dropped 18,744)
  Drop non-positive Quantity: 805,620 -> 805,620 (dropped 0)
  Drop non-positive Price: 805,620 -> 805,549 (dropped 71)
  Drop non-product stock codes: 805,549 -> 802,632 (dropped 2,917)
  Filter to country: 802,632 -> 724,440 (dropped 78,192)
  Cast Customer ID to int: 724,440 -> 724,440 (dropped 0)
  Add Revenue column: 724,440 -> 724,440 (dropped 0)
  Normalize string columns: 724,440 -> 724,440 (dropped 0)
  Split: 86 wholesale customers (99,850 rows), 5248 retail customers (624,590 rows)
  Drop non-positive-spend retail customers: 5248 -> 5248 customers
Final retail: 624,590 rows, 5,248 customers
Final wholesale: 99,850 rows, 86 customers


### Interpreting the cleaning log

Reading the audit trail from top to bottom:

- **243,007 rows dropped for missing Customer ID** (22.8% of raw data) — 
  matches the EDA finding. These transactions can't be linked to a 
  customer and so can't participate in segmentation.
- **18,744 cancellation invoices dropped** — slightly fewer than EDA's 
  ~22,950 negative-quantity rows, because step 1 already removed the 
  ~3,400 "internal adjustment" rows (which also had missing IDs).
- **71 non-positive-price rows dropped** — only 71 because most zero-price 
  entries also had missing IDs and were dropped earlier.
- **2,917 non-product stock codes dropped** — postage, bank charges, 
  Amazon fees, manual adjustments. Volume is small but they'd 
  contaminate revenue and product-breadth features.
- **78,192 non-UK rows dropped** — about 9.7% of post-prior-cleaning data, 
  consistent with the EDA finding that UK accounts for ~92% of revenue.

**The final split:** 86 wholesale customers, the rest retail.

**Reconciling with EDA's "103 wholesale customers":** the EDA count was 
on raw data; cleaning removed transactions (cancellations, non-UK, etc.) 
that pushed 17 borderline customers back under the 60-orders / £20k 
thresholds. Both numbers are correct — they describe the same threshold 
applied to data at different stages of cleaning. The post-cleaning 
count (86) is what we'll use going forward, because those are the 
customers actually held out from segmentation.

## 4 Sanity check the cleaned data

Quick verification:
- Do the dtypes look right? (Customer ID should be int, Revenue float, 
  InvoiceDate datetime64, Invoice/StockCode str)
- Do the values look sensible? (positive Quantity, positive Price, valid dates)
- Is the resulting DataFrame the expected shape?

In [10]:
print("\nRetail:")
print(retail.head())
print(f"\nDtypes:\n{retail.dtypes}")


Retail:
  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

          InvoiceDate  Price  Customer ID         Country  Revenue  
0 2009-12-01 07:45:00   6.95        13085  United Kingdom     83.4  
1 2009-12-01 07:45:00   6.75        13085  United Kingdom     81.0  
2 2009-12-01 07:45:00   6.75        13085  United Kingdom     81.0  
3 2009-12-01 07:45:00   2.10        13085  United Kingdom    100.8  
4 2009-12-01 07:45:00   1.25        13085  United Kingdom     30.0  

Dtypes:
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[n

In [11]:
# Reconcile with EDA findings
n_retail = retail['Customer ID'].nunique()
n_wholesale = wholesale['Customer ID'].nunique()
retail_revenue = retail['Revenue'].sum()
wholesale_revenue = wholesale['Revenue'].sum()
total_revenue = retail_revenue + wholesale_revenue

print(f"Post-cleaning customer counts:")
print(f"  Retail:    {n_retail:,} customers")
print(f"  Wholesale: {n_wholesale:,} customers")
print(f"  Total:     {n_retail + n_wholesale:,} customers")

print(f"\nPost-cleaning revenue:")
print(f"  Retail:    £{retail_revenue:>14,.0f}  ({retail_revenue/total_revenue:.1%})")
print(f"  Wholesale: £{wholesale_revenue:>14,.0f}  ({wholesale_revenue/total_revenue:.1%})")
print(f"  Total:     £{total_revenue:>14,.0f}")

Post-cleaning customer counts:
  Retail:    5,248 customers
  Wholesale: 86 customers
  Total:     5,334 customers

Post-cleaning revenue:
  Retail:    £     9,664,466  (66.1%)
  Wholesale: £     4,957,577  (33.9%)
  Total:     £    14,622,043


**Reconciled with EDA:**
- EDA (raw data): 5,942 customers → 103 wholesale + 5,839 retail; wholesale = 37% of revenue
- Post-cleaning: roughly 5,300 customers → 86 wholesale + ~5,250 retail; 
  wholesale revenue share recomputed above

The post-cleaning revenue share for wholesale is the figure that should 
appear in the final README — it reflects the cleaned data that 
downstream models actually see.

## 5 Save processed data

Write the cleaned retail and wholesale DataFrames to parquet. 
Parquet vs CSV/Excel for this kind of intermediate output:

- **~10x smaller** files (45 MB Excel → ~5 MB parquet)
- **~50x faster** to load (60s Excel → ~1s parquet)
- **Preserves dtypes** (no need to re-parse dates or recast Customer ID 
  to int each load)

Downstream notebooks load these parquets directly and never touch 
the Excel file again.

In [12]:
retail.to_parquet("../data/processed/retail_clean.parquet", index=False)
wholesale.to_parquet("../data/processed/wholesale_clean.parquet", index=False)
print("Saved cleaned data to data/processed/")

Saved cleaned data to data/processed/


## Cleaning Summary

Applied `src.preprocessing.clean_transactions()` to the raw 1.07M-row 
transaction file. The audit trail above shows row counts after each step.

### Outputs

Two parquet files in `data/processed/`:

- **`retail_clean.parquet`** — primary dataset for segmentation and CLV. 
  ~[N] customers, ~[N] transactions, all UK, all positive-revenue.
- **`wholesale_clean.parquet`** — held-out dataset of 86 wholesale 
  accounts for separate top-accounts reporting.

### Headline numbers (post-cleaning)

- **Retail customers:** 5,248
- **Wholesale customers:** 86
- **Wholesale share of post-cleaning revenue:** £4,957,577  (33.9%)

### Key cleaning decisions (and rationale)

| Decision | Reason |
|---|---|
| Drop rows with missing Customer ID | Cannot segment unidentified customers; 22% of raw |
| Drop "C"-prefix invoices | Cancellations are returns, not sales |
| Drop non-product StockCodes | Postage, fees, adjustments — not customer purchases |
| Filter to UK only | 92% of revenue; multi-country adds noise without insight |
| Split wholesale at >60 orders OR >£20k | Practical cutoff (no natural gap in distribution); preserves actionable retail segmentation |
| Drop non-positive-spend retail customers | Final safety net — customers whose returns exceed purchases aren't customers in the segmentation sense |

### Caveat documented for the README

The wholesale/retail split is a **practical convention**, not a 
discovered natural break. Different thresholds (e.g., 100 orders / £50k) 
would shift a handful of customers between the two groups. The current 
choice is justified in the EDA notebook by the absence of a visual 
gap in the order-count × spend distribution.

→ Proceed to notebook **03_feature_engineering.ipynb**